# 02 · Etiquetado por reglas con PySpark: intención, categoría, urgencia y frustración

**Objetivo:** leer el archivo Parquet generado por el notebook 01 directamente con Spark (`spark.read.parquet`), asignar a cada tweet de **cliente** sus etiquetas automáticas mediante funciones nativas de PySpark (`F.when`, `F.rlike`, `F.greatest`) y exportar tanto el dataset completo etiquetado en formato Parquet como la muestra aleatoria estratificada para validación manual.

| Archivo generado | Para qué sirve |
|---|---|
| `../data/processed/clientes_etiquetados.parquet` | **Todos** los tweets de clientes con la etiqueta automática de las reglas. |
| `../data/processed/muestra_para_etiquetar.csv` | Muestra estratificada **sin** etiquetas automáticas, para etiquetar a mano a ciegas. |
| `../data/processed/muestra_clave_reglas.csv` | Los mismos tweets de la muestra **con** la etiqueta automática; se usa al final para calcular el F1-score. |

## 1. Preparación del entorno y lectura directa de Parquet con Spark
Inicializamos la sesión de Spark 3.5.9 siguiendo la configuración de los laboratorios y leemos `apple_support_filtrado.parquet` directamente como DataFrame de Spark.

In [ ]:
# ============================================================
# 1. PREPARACIÓN DE SPARK Y LECTURA DE PARQUET
# ============================================================
import os
import sys
from pathlib import Path
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import classification_report

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark import StorageLevel

python_executable = sys.executable
os.environ["PYSPARK_PYTHON"] = python_executable
os.environ["PYSPARK_DRIVER_PYTHON"] = python_executable
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"

RUTA_IN = Path("../data/processed/apple_support_filtrado.parquet")
RUTA_OUT = Path("../data/processed/clientes_etiquetados.parquet")
RUTA_MUESTRA = Path("../data/processed/muestra_para_etiquetar.csv")
RUTA_CLAVE = Path("../data/processed/muestra_clave_reglas.csv")

assert RUTA_IN.exists(), f"No se encontró el archivo Parquet de entrada: {RUTA_IN.resolve()}"

spark = (
    SparkSession.builder
    .appName("Tesis_02_Clasificacion_Reglas")
    .master("local[*]")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print("Spark:", spark.version, "| Spark UI:", spark.sparkContext.uiWebUrl)

# Lectura directa del Parquet en Spark (estilo Laboratorio 7 y 8)
df_filtrado = spark.read.parquet(str(RUTA_IN))
df_filtrado.printSchema()

df_clientes = (
    df_filtrado
    .filter(F.col("rol") == "cliente")
    .select("tweet_id", "rol", "fecha", "texto_limpio")
)
print("Total de tweets de clientes:", f"{df_clientes.count():,}")

## 2. Catálogo de expresiones regulares
Cada lista de patrones se compila en una expresión regular case-insensitive (`(?i)\b(?:...)`) compatible con el motor `rlike` de Spark SQL.

In [ ]:
# ============================================================
# 2. CATÁLOGO DE EXPRESIONES REGULARES PARA SPARK (RLIKE)
# ============================================================
FALLA_FUERTE = [
    r"not working", r"stopped working", r"not able", r"unable", r"won'?t", r"can'?t",
    r"cannot", r"couldn'?t", r"no longer", r"broken", r"crash", r"bug", r"error",
    r"stuck", r"freez", r"frozen", r"glitch", r"lag", r"fail", r"outage", r"down\b",
    r"slow", r"drain", r"dying", r"died", r"dies\b", r"lost", r"missing", r"disappear",
    r"black screen", r"unresponsive", r"soft reset", r"hard reset", r"force restart",
    r"fallen", r"fell\b", r"cracked", r"overheat", r"ruin",
    r"(?:isn|aren|wasn|weren|doesn|don|didn|hasn|haven|hadn)'?t\s+(?:\w+\s+){0,2}?"
    r"(?:allow\w*|let\w*|work\w*|respond\w*|load\w*|show\w*|charg\w*|connect\w*|sync\w*|open\w*|turn\w*)",
    r"not\s+(?:receiving|sending|syncing|charging|loading|showing|allowing|letting|working|responding)"
]

FALLA_DEBIL = [
    r"issue", r"problem", r"fix", r"restart", r"reboot", r"reset", r"keeps",
    r"annoying", r"weird", r"wrong"
]

SOLICITUD_EXPLICITA = [
    r"how (?:do|can|to|does|should)", r"is it possible", r"is there a way"
]

SOLICITUD = [
    r"where (?:do|can|is|are)", r"what (?:is|does|are|happens)", r"when (?:will|is|does)",
    r"is there", r"does (?:anyone|apple|it|the)", r"do (?:you|apple)", r"can (?:i|you)",
    r"could (?:you|i)", r"any (?:way|idea|tips?|update|news)", r"please (?:help|advise)",
    r"how much", r"which", r"wondering", r"want to know", r"looking for", r"need (?:help|to)",
    r"i want to", r"would like to", r"trying to"
]

CATEGORIAS = {
    "telefonia_llamadas": [
        r"(?:phone |voice )?calls?\b", r"phone calls?", r"voicemail",
        r"dial", r"ringing", r"no service", r"\bcaller"
    ],
    "cuenta_icloud": [
        r"icloud", r"apple id", r"password", r"log ?in", r"sign ?in", r"locked",
        r"verification", r"2fa", r"backup", r"storage", r"account"
    ],
    "pagos_facturacion": [
        r"charged", r"charges?\b", r"billing", r"\bbills?\b", r"refund",
        r"payment", r"subscription", r"invoice", r"apple pay",
        r"credit card", r"purchase"
    ],
    "servicios_apple": [
        r"apple music", r"imessage", r"facetime", r"siri", r"icloud mail",
        r"apple tv", r"podcast", r"maps\b", r"itunes", r"airdrop",
        r"homepod", r"carplay"
    ],
    "conectividad": [
        r"wi-?fi", r"bluetooth", r"cellular", r"signal", r"network",
        r"mobile data", r"data\b", r"sim\b", r"carrier", r"roaming",
        r"hotspot", r"lte\b", r"internet"
    ],
    "apps_funciones": [
        r"app store", r"apps?\b", r"download", r"testflight", r"safari",
        r"mail\b", r"photos?\b", r"widget", r"notification", r"feature",
        r"quick edit", r"edit\b"
    ],
    "ios_software_rendimiento": [
        r"ios(?:\s?\d+)?\b", r"update", r"upgrade", r"macos", r"beta",
        r"install", r"firmware", r"keyboard", r"glitch", r"bug",
        r"freez", r"frozen", r"crash", r"lag", r"slow",
        r"unresponsive", r"performance"
    ],
    "hardware_dispositivo": [
        r"battery", r"screens?\b", r"charging", r"charger", r"cracked",
        r"fallen", r"fell\b", r"dropped", r"water", r"speaker",
        r"microphone", r"buttons?\b", r"camera", r"overheat",
        r"turn(?:ing)? on", r"display", r"touch id", r"face id"
    ]
}

DISPOSITIVO = [
    r"iphone", r"ipad", r"ipod", r"mac(?:book)?\b", r"imac", r"phone",
    r"(?:apple )?watch\b", r"airpods", r"device", r"laptop", r"tablet"
]

CRITICA = [
    r"outage", r"down for", r"(?:is|are) down\b", r"services? (?:is |are )?down",
    r"completely (?:dead|broken|unusable|died)", r"unusable", r"bricked",
    r"won'?t (?:turn on|power|boot|start)", r"boot ?loop", r"stuck on (?:the )?(?:apple )?logo",
    r"data loss", r"lost (?:all|everything|my (?:photos|data|contacts|messages))",
    r"(?:photos|data|contacts|messages) (?:gone|deleted|disappeared|lost)",
    r"hacked", r"unauthori[sz]ed", r"stolen", r"fraud", r"locked out",
    r"(?:can'?t|cannot|unable to|not able to)\s+(?:log ?in|sign ?in|access)",
    r"charged (?:twice|\d+)", r"double charged", r"disabled", r"no service", r"911", r"emergency",
    r"(?:not able to|unable to|can'?t|cannot|couldn'?t)\s+(?:\w+\s+){0,3}?(?:make|receive)\s+(?:\w+\s+){0,2}?calls?",
    r"(?:fell|fallen|dropped|water|wet)[^.!?]{0,40}(?:stopped working|dead|not working|won'?t turn on)"
]

MEDIA_EXTRA = [
    r"problem", r"issue", r"fix", r"keeps", r"restart", r"reboot", r"reset"
]

AMPLIFICADORES = [
    r"urgent", r"asap", r"immediately", r"still\b", r"for (?:days|hours|weeks|months)",
    r"since (?:yesterday|monday|tuesday|wednesday|thursday|friday|this morning|last)",
    r"(?:past|last) (?:\w+ ){0,2}(?:hours?|days?|weeks?)", r"twice", r"(?:three|four|multiple) times",
    r"again", r"constantly", r"every (?:time|day|hour)", r"all the time", r"everyone",
    r"anyone else", r"all (?:users|of us)", r"for work", r"work(?:ing)? from", r"business"
]

FRUSTRACION = [
    r"fuck", r"shit", r"damn", r"crap", r"ridiculous", r"unacceptable",
    r"worst", r"horrible", r"frustrat", r"angry", r"switch(?:ing)? to",
    r"never (?:buy|again)", r"🤬", r"😡", r"😤"
]

def patron(lista):
    return r"(?i)\b(?:" + "|".join(lista) + r")"

RX_FALLA_FUERTE = patron(FALLA_FUERTE + CRITICA)
RX_FALLA_DEBIL = patron(FALLA_DEBIL)
RX_SOL_EXPLICITA = patron(SOLICITUD_EXPLICITA)
RX_SOLICITUD = patron(SOLICITUD)
RX_CRITICA = patron(CRITICA)
RX_MEDIA = patron(FALLA_FUERTE + MEDIA_EXTRA)
RX_AMPL = patron(AMPLIFICADORES)
RX_FRUSTRACION = patron(FRUSTRACION)
RX_DISPOSITIVO = patron(DISPOSITIVO)

## 3. Función de clasificación distribuida en PySpark
Construimos las columnas `intencion`, `categoria`, `urgencia` y `frustracion` utilizando únicamente transformaciones nativas de Spark (`F.when`, `F.rlike`, `F.greatest`), evitando UDFs en Python para que Catalyst optimice el plan físico.

In [ ]:
# ============================================================
# 3. CLASIFICACIÓN CON FUNCIONES NATIVAS DE SPARK (F)
# ============================================================
def clasificar_spark(df, col_texto="texto_limpio"):
    t = F.regexp_replace(F.coalesce(F.col(col_texto), F.lit("")), r"[’‘`]", "'")
    df_temp = df.withColumn("_t", t)

    # 1. Intención
    cond_fuerte = F.col("_t").rlike(RX_FALLA_FUERTE)
    cond_explicita = F.col("_t").rlike(RX_SOL_EXPLICITA)
    cond_debil = F.col("_t").rlike(RX_FALLA_DEBIL)
    cond_solicitud = F.col("_t").rlike(RX_SOLICITUD) | F.col("_t").contains("?")

    col_intencion = (
        F.when(cond_fuerte, F.lit("falla"))
        .when(cond_explicita, F.lit("solicitud"))
        .when(cond_debil, F.lit("falla"))
        .when(cond_solicitud, F.lit("solicitud"))
        .otherwise(F.lit("otro"))
    )
    df_temp = df_temp.withColumn("intencion", col_intencion)

    # 2. Categoría (conteo de coincidencias por categoría; desempate por orden del catálogo)
    cols_puntaje = []
    for cat, lista_patrones in CATEGORIAS.items():
        expr_suma = F.lit(0)
        for p in lista_patrones:
            expr_suma = expr_suma + F.when(F.col("_t").rlike(patron([p])), 1).otherwise(0)
        nombre_col = f"_score_{cat}"
        df_temp = df_temp.withColumn(nombre_col, expr_suma)
        cols_puntaje.append((cat, nombre_col))

    df_temp = df_temp.withColumn(
        "_max_score",
        F.greatest(*[F.col(c) for _, c in cols_puntaje])
    )

    expr_cat = F.when(
        F.col("_t").rlike(RX_DISPOSITIVO),
        F.lit("dispositivo_general")
    ).otherwise(F.lit("otros"))

    for cat, nombre_col in reversed(cols_puntaje):
        expr_cat = F.when(
            (F.col("_max_score") > 0) & (F.col(nombre_col) == F.col("_max_score")),
            F.lit(cat)
        ).otherwise(expr_cat)

    df_temp = df_temp.withColumn("categoria", expr_cat)

    # 3. Urgencia (solo aplica para intención == 'falla')
    es_falla = F.col("intencion") == "falla"
    cond_critica = F.col("_t").rlike(RX_CRITICA)
    cond_media = F.col("_t").rlike(RX_MEDIA)
    cond_ampl = F.col("_t").rlike(RX_AMPL)

    col_urgencia = (
        F.when(~es_falla, F.lit("baja"))
        .when(cond_critica | (cond_media & cond_ampl), F.lit("critica"))
        .when(cond_media, F.lit("media"))
        .otherwise(F.lit("baja"))
    )
    df_temp = df_temp.withColumn("urgencia", col_urgencia)

    # 4. Frustración
    df_temp = df_temp.withColumn("frustracion", F.col("_t").rlike(RX_FRUSTRACION))

    cols_borrar = ["_t", "_max_score"] + [c for _, c in cols_puntaje]
    return df_temp.drop(*cols_borrar)

## 4. Prueba con ejemplos de verificación
Verificamos el comportamiento de las reglas sobre casos reales y de contraste antes de procesar todo el conjunto de datos.

In [ ]:
# ============================================================
# 4. PRUEBA CON EJEMPLOS
# ============================================================
ejemplos = [
    ("@AppleSupport my phone is not able to receive or make any phone calls.",),
    ("@AppleSupport iphone glitch",),
    ("@AppleSupport mac glitch",),
    ("I've had to soft reset my phone twice in the past two hours because it became unresponsive",),
    ("@AppleSupport please fix the fucking problem with keyboard or I'll switch to @USER",),
    ("@AppleSupport since yesterdays update, my phone is dying of glitches and constant freezes",),
    ("@AppleSupport It‘s fallen down and then stopped working",),
    ("Hey @AppleSupport the iOS update is ruining my Apple iPhone experience pls fix",),
    ("@AppleSupport No. And it also isn't allowing me to do the new quick edit thing. Help?",),
    ("@AppleSupport how do I restart my iPhone?",),
    ("@AppleSupport thank you so much, that worked!",)
]

df_ejemplos = spark.createDataFrame(ejemplos, ["texto_limpio"])
display(clasificar_spark(df_ejemplos).toPandas())

## 5. Aplicación al dataset completo, plan físico y distribuciones
Aplicamos las reglas al DataFrame de clientes, inspeccionamos el plan físico con `explain("formatted")`, persistimos el resultado y calculamos las distribuciones con agregaciones de Spark.

In [ ]:
# ============================================================
# 5. APLICACIÓN AL DATASET COMPLETO Y DISTRIBUCIONES EN SPARK
# ============================================================
df_etiquetado = clasificar_spark(df_clientes)
df_etiquetado.persist(StorageLevel.MEMORY_AND_DISK)

print("PLAN FÍSICO DE CLASIFICACIÓN:")
df_etiquetado.explain("formatted")

total_clientes = df_etiquetado.count()
print("Total de tweets de clientes etiquetados:", f"{total_clientes:,}")

for dim in ["intencion", "categoria", "urgencia", "frustracion"]:
    print(f"\n=== DISTRIBUCIÓN: {dim.upper()} ===")
    (
        df_etiquetado
        .groupBy(dim)
        .agg(F.count("*").alias("count"))
        .withColumn("pct", F.round(F.col("count") / F.lit(total_clientes), 3))
        .orderBy(F.desc("count"))
        .show(truncate=False)
    )

print("\n=== TABLA CRUZADA: CATEGORÍA vs URGENCIA (SOLO FALLAS) ===")
(
    df_etiquetado
    .filter(F.col("intencion") == "falla")
    .groupBy("categoria")
    .pivot("urgencia", ["baja", "media", "critica"])
    .count()
    .na.fill(0)
    .orderBy("categoria")
    .show(truncate=False)
)

## 6. Guardado en Parquet y diagnóstico de cobertura de categorías
Llevamos el resultado etiquetado al Driver con `toPandas()` para almacenarlo en Parquet con PyArrow y revisamos los n-gramas más frecuentes en las fallas sin categoría específica (`otros` o `dispositivo_general`).

In [ ]:
# ============================================================
# 6. GUARDADO EN PARQUET Y REVISIÓN DE COBERTURA
# ============================================================
cl = df_etiquetado.toPandas()
RUTA_OUT.parent.mkdir(parents=True, exist_ok=True)
cl.to_parquet(RUTA_OUT, index=False, engine="pyarrow", compression="snappy")
print("Guardado en:", RUTA_OUT.resolve(), "| Filas:", len(cl))

fallas = cl[cl["intencion"] == "falla"]
sin_tema = fallas["categoria"].isin(["otros", "dispositivo_general"])
print(f"Fallas sin tema específico ('otros' + 'dispositivo_general'): {sin_tema.mean():.1%}")

vec = CountVectorizer(stop_words="english", ngram_range=(1, 2), min_df=20)
X = vec.fit_transform(fallas.loc[sin_tema, "texto_limpio"])
top = sorted(zip(vec.get_feature_names_out(), X.sum(axis=0).A1), key=lambda x: -x[1])[:30]
display(pd.DataFrame(top, columns=["termino", "frecuencia"]))

## 7. Muestra estratificada para etiquetado manual (a ciegas)
Extraemos hasta 40 tweets por combinación `intencion × urgencia` y exportamos `muestra_para_etiquetar.csv` (sin etiquetas automáticas) y `muestra_clave_reglas.csv` (con las etiquetas de las reglas).

In [ ]:
# ============================================================
# 7. GENERACIÓN DE MUESTRA PARA ETIQUETADO MANUAL
# ============================================================
muestra = (
    cl.sample(frac=1, random_state=42)
    .groupby(["intencion", "urgencia"], group_keys=False)
    .head(40)
    .copy()
)

para_etiquetar = muestra[["tweet_id", "texto_limpio"]].copy()
for c in ["intencion_manual", "categoria_manual", "urgencia_manual", "comentario_error"]:
    para_etiquetar[c] = ""

RUTA_MUESTRA.parent.mkdir(parents=True, exist_ok=True)
para_etiquetar.to_csv(RUTA_MUESTRA, index=False, encoding="utf-8-sig")
muestra[["tweet_id", "intencion", "categoria", "urgencia"]].to_csv(RUTA_CLAVE, index=False)
print(len(muestra), "tweets exportados en la muestra:", RUTA_MUESTRA.resolve())

## 8. Evaluación contra etiquetado manual y cierre de Spark
Una vez completado `muestra_para_etiquetar.csv`, esta celda calcula precisión, recall y **F1-score** por dimensión. Finalmente liberamos la caché y detenemos la `SparkSession`.

In [ ]:
# ============================================================
# 8. EVALUACIÓN F1-SCORE Y CIERRE DE SPARK
# ============================================================
manual = pd.read_csv(RUTA_MUESTRA, dtype={"tweet_id": str}).fillna("")
clave = pd.read_csv(RUTA_CLAVE, dtype={"tweet_id": str})
m = manual.merge(clave, on="tweet_id")
m = m[m["intencion_manual"].str.strip() != ""]

if m.empty:
    print("Aún no hay etiquetas manuales en muestra_para_etiquetar.csv.")
else:
    for dim in ["intencion", "categoria", "urgencia"]:
        y_true = m[f"{dim}_manual"].str.strip().str.lower()
        y_pred = m[dim].str.strip().str.lower()
        print(f"\n=== {dim.upper()} (n={len(m)}) ===")
        print(classification_report(y_true, y_pred, digits=3, zero_division=0))

df_etiquetado.unpersist(blocking=True)
spark.stop()
print("SparkSession detenida.")